[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/eight_conics/eight_conics.ipynb)

# Penrose's Eight-Conic Theorem

This example computes Penrose's eight-conic theorem, which completes a pattern of quadrics touching in pairs. Two quadrics in double contact touch at two points with common tangents there. Arranged on the corners of a cube, with neighbours in double contact and the four contact planes of every face meeting in a point, seven such quadrics admit an eighth that touches its three neighbours in double contact as well.

A quadric is modelled as a map from points to their polar planes, `Quadric = Plane <- Point`, which takes a point of the curve to its tangent. Adding the dyad of a plane, `contact_plane * (contact_plane & Point)`, changes the quadric everywhere except on that plane, so the new quadric meets the plane in the same points with the same tangents: double contact is an addition of maps. The eighth quadric follows from each of its three neighbours by adding one more contact dyad, and the theorem is that the three sums are the same map.

In matrix notation a quadric reads as a symmetric $3 \times 3$ matrix $C$, double contact along the plane $\ell$ as the pencil $C + \lambda\,\ell\ell^{\mathsf T}$, and the theorem as three such pencils through the eighth quadric's matrix.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation

from examples.math.eight_conics import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Quadric = ga.gatype((Plane, Point))
DualQuadric = ga.gatype((Point, Plane))

## 1. Double contact

A sphere with the dyad of a plane added, `sphere + strength * contact_plane * (contact_plane & Point)`, is another quadric, which agrees with the sphere at every point of the plane, since there the dyad pairs to nothing. The two meet the plane in the same points with the same polar planes, the tangents there, whatever the strength: they are in double contact. In the figure the sphere is blue and the quadric orange, through the same two points of their dashed contact_plane.

In [ ]:
# The strength of the dyad, and the contact plane: off the sphere's centre, and weighted.
strength = 2.5
plane_offset = 0.5
plane_weight = 0.7

center = mv.w.dual()                                                            # Point-
semi_axes = stack((mv.x.dual(), mv.y.dual()))                             # [2] Point-
contact_plane = (mv.x - mv.w * plane_offset) * plane_weight                     # Plane+

In [ ]:
def intersections(quadrics: Quadric, planes: Plane) -> Point:
    """The two real crossings of each quadric with its plane, on the last batch axis."""
    origin = mv.w.dual()
    planes = planes.normalized()
    # Reflection gives the foot of the perpendicular from the chart origin.
    anchor = (origin + (planes >> origin)) / 2                              # [...] Point-
    direction = (planes | mv.xy).dual()                                     # [...] Point-
    quadratic = quadrics(direction) & direction                             # [...] Scalar+
    linear = quadrics(anchor) & direction                                   # [...] Scalar+
    constant = quadrics(anchor) & anchor                                    # [...] Scalar+
    root_signs = np.array([-1, 1])
    distances = (-linear[..., None]
                 + (linear.squared() - quadratic * constant).square_root()[..., None]
                 * root_signs) / quadratic[..., None]                       # [..., 2] Scalar+
    return anchor[..., None] + direction[..., None] * distances               # [..., 2] Point-


# The dual quadric maps tangent planes to their points of contact.
dual_sphere = ((semi_axes * (Plane & semi_axes)).sum(axis=0)                # DualQuadric
                         - center * (Plane & center))                    # [] Point- <- Plane+
sphere = dual_sphere.inverse()                                              # [] Plane+ <- Point-

# This dyad pairs to nothing with every point of the contact plane: added, it keeps the sphere's two crossings
# with the contact plane and their polar planes.
double_plane = contact_plane * (contact_plane & Point)                      # [] Plane+ <- Point-
touching = sphere + strength * double_plane                                 # [] Plane+ <- Point-
contacts = intersections(sphere, contact_plane)                             # [2] Point-
print("the contacts on the other quadric:", (touching(contacts) & contacts).to_array())
print("the tangents there, less the sphere's:", (touching(contacts) - sphere(contacts)).norm().to_array())
save_figure(render.draw_double_contact(sphere, touching, contact_plane, contacts), "eight_conics_double_contact");

## 2. Compatible contacts

The quadrics occupy the vertices of a cube whose edges record double contact. Each face must also have its four contact planes pass through one point. The cube describes the relationships between quadrics in space. Three contact_plane choices and three coupling parameters build seven quadrics satisfying these conditions.

In [ ]:
# The edges' three contact planes, by their turn and their offset from the centre, and their weight: strong contact planes
# make the eight quadrics distinct ellipsoids, stronger ones turn some into hyperboloids. The three face
# couplings, swept around a loop a third of a turn apart, the frames around it and their duration.
plane_angles = np.arange(3) * (2 * np.pi / 3)
plane_offsets = np.array([0.20, 0.24, 0.16])
cube_plane_weight = 1.3
mean_couplings = np.array([0.30, -0.24, 0.27])
coupling_amplitude = 0.25
n_frames = 96
duration_ms = 90
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
couplings = mean_couplings + coupling_amplitude * np.cos(phases[:, None] + plane_angles)

# Each face pairs the other two contact planes.
forward = np.array([1, 2, 0])
backward = np.array([2, 0, 1])
# Binary vertex labels differ by one bit along each edge.
edges = np.array([[0, 1], [0, 2], [0, 4], [1, 3], [2, 3], [1, 5],
                  [4, 5], [2, 6], [4, 6], [3, 7], [5, 7], [6, 7]])
# The edges joining the eighth quadric to its three neighbours.
closing_edges = np.array([9, 10, 11])

normals = (mv.xy * (-plane_angles / 2)).exp() >> mv.x                     # [3] Plane+
contact_planes = (normals - mv.w * plane_offsets) * cube_plane_weight     # [3] Plane+

In [ ]:
double_planes = contact_planes * (contact_planes & Point)                # [3] Plane+ <- Point-
neighbours = sphere + double_planes                                      # [3] Plane+ <- Point-
left = contact_planes[forward]                                           # [3] Plane+
right = contact_planes[backward]                                         # [3] Plane+
face_scales = 1 - couplings ** 2

# The cross terms couple two contact-plane choices. Each resulting quadric
# touches both neighbours, with all four contact planes concurrent.
cross_terms = left * (right & Point) + right * (left & Point)            # [3] Plane+ <- Point-
faces = (sphere * face_scales + double_planes[forward]                   # Quadric
               + double_planes[backward] + cross_terms * couplings)      # [frames, 3] Plane+ <- Point-
left_planes = left * couplings + right                                   # [frames, 3] Plane+
right_planes = left + right * couplings                                  # [frames, 3] Plane+

## 3. The eighth quadric

The remaining three edges must reach the same quadric. Each quadric adjacent to the missing vertex supplies a candidate, obtained by adding a contact dyad with the appropriate scale. All three candidates agree throughout the deformation.

In [ ]:
closing_scale = 1 - np.sum(couplings ** 2, axis=-1) - 2 * np.prod(couplings, axis=-1)
closing_planes = (                                                              # Plane+
    contact_planes * face_scales
    + left * (couplings[:, backward] + couplings * couplings[:, forward])
    + right * (couplings[:, forward] + couplings * couplings[:, backward])
)                                                                      # [frames, 3] Plane+

# Each candidate is in double contact with its corresponding face quadric.
# Agreement of the three maps closes all three remaining edges at once.
completions = ((faces * closing_scale[:, None]                                  # Quadric
                      + closing_planes * (closing_planes & Point))
                     / face_scales)                                    # [frames, 3] Plane+ <- Point-
eighth = completions[:, 0]                                              # [frames] Plane+ <- Point-

quadrics = stack((sphere.broadcast_to((n_frames,)),                             # Quadric
    neighbours[0].broadcast_to((n_frames,)), neighbours[1].broadcast_to((n_frames,)),
    faces[:, 2], neighbours[2].broadcast_to((n_frames,)), faces[:, 1], faces[:, 0],
    eighth), axis=-1)                                                  # [frames, 8] Plane+ <- Point-
edge_planes = stack((contact_planes[0].broadcast_to((n_frames,)),               # Plane+
    contact_planes[1].broadcast_to((n_frames,)), contact_planes[2].broadcast_to((n_frames,)),
    left_planes[:, 2], right_planes[:, 2], right_planes[:, 1], left_planes[:, 1],
    left_planes[:, 0], right_planes[:, 0], closing_planes[:, 2],
    closing_planes[:, 1], closing_planes[:, 0]), axis=-1)                       # [frames, 12] Plane+
contacts = intersections(quadrics[:, edges[:, 0]], edge_planes)                 # [frames, 12, 2] Point-
show_animation(render.animate_eighth(quadrics, edge_planes, contacts, edges, closing_edges), "eight_conics_eighth",
               duration_ms)

The eighth quadric is drawn bold in orange and its three neighbours in blue, green and pink, each touching it at two points of their dashed common contact_plane; the other four quadrics are faint. As the couplings change, all eight quadrics move, and the eighth keeps all six of its contacts.

In [ ]:
# Agreement as maps checks every input point, not only the drawn contacts.
print("Eighth quadric")
print(eighth[0].coefficient_table())
print("Differences from the other two constructions")
for difference in completions[0, 1:] - eighth[0]:
    print(difference.coefficient_table())

In [ ]:
# checks
# The three constructions of the eighth quadric are one map, at every coupling of the deformation.
np.testing.assert_allclose((completions[:, 1:] - completions[:, :1]).kernel, 0, atol=1e-12)

The closure is an identity between sums of contact dyads: the same map carries all three tangency requirements. The construction follows the Penrose formulas in §6 of Arnold, Chern, Eide, Gunn, Neukirchner and Penrose's [*Penrose's eight-conic theorem*](https://arxiv.org/html/2409.17150v8#S6). Their theorem also covers configurations beyond the ellipsoids shown here.